# 第九轮完成结果的经济解释

只读最终Qlib Recorder；完整引用未封存时立即退出。没有新拟合、目标生成、策略回放或门槛修改。

In [1]:
"""Finished-Recorder economic interpretation only; no fit, target or path generation."""
from pathlib import Path
import json,hashlib
import numpy as np,pandas as pd
import qlib
from qlib.workflow import R
ROOT=Path('/Users/rziiiii/dev/workspace-worktrees/feasible-parent-sparse-search-20261003')
CARD=ROOT/'research/feasible_parent_sparse_search_20261003'
reference=json.loads((CARD/'market_reference.json').read_text())
if reference.get('complete') is not True:raise SystemExit('Market reference incomplete; no economic conclusions computed.')
plan=json.loads((CARD/'selection_plan.json').read_text())
qlib.init(provider_uri=str(ROOT/plan['source']['provider_uri']),region='us',kernels=1)
identities={}
with R.uri_context(reference['recorder_uri']):
 rec=R.get_recorder(experiment_name=plan['experiment'],recorder_id=reference['recorder_id'])
 assert rec.client.get_run(rec.id).info.status=='FINISHED'
 def get(key):
  entry=reference['artifacts'][key];path=Path(rec.get_local_dir())/'artifacts'/entry['artifact']
  assert path.stat().st_size==entry['bytes'] and hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256']
  value=rec.load_object(entry['artifact']);assert hashlib.sha256(path.read_bytes()).hexdigest()==entry['sha256']
  identities[key]=entry;return value
 summary=get('summary')
 names=['comparison','annual','gates','bootstrap','seed_metrics','attribution','window_metrics','paired_parent_comparison',
        'execution_sparsity','saved_path_fee_decomposition','seed_path_equality','training_diagnostics','ensemble_audits','portfolio_replays']
 objects={name:get('analysis/'+name+'.pkl') for name in names}
def rows(frame):return json.loads(frame.to_json(orient='records',date_format='iso',double_precision=15))
paths=objects['portfolio_replays'];events=[];annual_magnitudes=[]
for (segment,variant,cost),part in paths.loc[paths.seed.eq('ensemble')].groupby(['segment','variant','cost_bps']):
 part=part.sort_values('signal_date').reset_index(drop=True)
 ref=paths.loc[paths.segment.eq(segment)&paths.seed.eq('benchmark')&paths.cost_bps.eq(cost)].sort_values('signal_date').reset_index(drop=True)
 assert part[['signal_date','trade_date','exit_date']].equals(ref[['signal_date','trade_date','exit_date']])
 excess=np.log1p(part.strategy_return.to_numpy())-np.log1p(ref.strategy_return.to_numpy())
 totals=float(excess.sum());first=float(excess[0]);best=int(np.argmax(excess));worst=int(np.argmin(excess))
 identity={'segment':segment,'variant':variant,'cost_bps':int(cost)}
 events.append({**identity,'sessions':len(part),'first_trade':str(part.trade_date.iloc[0].date()),'last_exit':str(part.exit_date.iloc[-1].date()),
  'total_net_log_excess':totals,'first_observation_log_excess':first,'remaining_saved_observations_log_excess':float(excess[1:].sum()),
  'largest_positive_observation_log_excess':float(excess[best]),'largest_positive_observation_trade_date':str(part.trade_date.iloc[best].date()),
  'largest_negative_observation_log_excess':float(excess[worst]),'largest_negative_observation_trade_date':str(part.trade_date.iloc[worst].date()),
  'exactly_equal_benchmark_return_days':int((part.strategy_return.to_numpy()==ref.strategy_return.to_numpy()).sum()),
  'max_absolute_daily_net_log_difference':float(abs(excess).max()),
  'mean_soxl':float(part.soxl_weight.mean()),'mean_soxx':float(part.soxx_weight.mean()),'mean_cash':float(part.cash_weight.mean()),
  'exact_soxx_target_days':int(((part.soxl_weight==0)&(part.soxx_weight==1)&(part.cash_weight==0)).sum()),
  'scope':'Additive attribution on the saved book, not a restarted or counterfactual strategy.'})
 for year,index in part.groupby(part.signal_date.dt.year).groups.items():
  index=np.array(list(index));a=excess[index]
  annual_magnitudes.append({**identity,'signal_year':int(year),'net_log_excess':float(a.sum()),
   'first_model_year_observation_log_excess':float(a[0]),'remaining_saved_year_log_excess':float(a[1:].sum()),
   'largest_absolute_daily_log_excess':float(abs(a).max()),
   'absolute_total_below_existing_aggregate_numerical_guard_1e_minus8':bool(abs(a.sum())<1e-8),
   'annotation_scope':'Magnitude-only use of existing aggregate1e-8 numerical guard; no annual gate was changed.'})
training=pd.DataFrame(objects['training_diagnostics']);ensembles=pd.DataFrame(objects['ensemble_audits'])
training_summary=[]
for variant,part in training.groupby('variant'):
 development=part[part.year.le(2022)]
 training_summary.append({'variant':variant,'all_fits':len(part),'development_fits':len(development),
  'development_selected_origins':development.selected_origin.value_counts().to_dict(),
  'development_F_delta_min':float(development.F_delta_from_parent.min()),'development_F_delta_max':float(development.F_delta_from_parent.max()),
  'development_F_delta_median':float(development.F_delta_from_parent.median()),
  'development_failures':int(development.failure.notna().sum()),
  'development_unique_two_cost_evaluations':sum(a['unique_full_path_evaluations'] for a in development.optimizer_diagnostics)})
report={'scope':'Read-only interpretation of completed market evidence; previously reviewed history, not fresh forward.',
 'recorder_id':reference['recorder_id'],'market_reference_sha256':hashlib.sha256((CARD/'market_reference.json').read_bytes()).hexdigest(),
 'source_artifacts':identities,'official_selected':summary['selected'],'official_historical_qualification':summary['historical_qualification_pass'],
 'new_forward_qualified':summary['new_forward_qualified'],'objective_achieved':summary['objective_achieved'],'parent_work':summary['parent_work'],
 'comparison':rows(objects['comparison']),'annual':rows(objects['annual']),'official_gates':rows(objects['gates']),
 'bootstrap':rows(objects['bootstrap']),'seed_metrics':rows(objects['seed_metrics']),
 'attribution':rows(objects['attribution']),'window_metrics':rows(objects['window_metrics']),
 'paired_parent_comparison':rows(objects['paired_parent_comparison']),'execution_sparsity':rows(objects['execution_sparsity']),
 'gross_fee_decomposition':rows(objects['saved_path_fee_decomposition']),
 'seed_path_equality':rows(objects['seed_path_equality']),'training_summary':training_summary,
 'ensemble_summary':rows(ensembles[[c for c in ['variant','year','selected_origin','candidate_ensemble_F','parent_ensemble_F','candidate_minus_parent_F','failure','candidate_component_origins'] if c in ensembles]]),
 'saved_book_contribution_diagnostics':events,'annual_magnitude_annotations':annual_magnitudes}
Path('/tmp/phase9_economic_audit.json').write_text(json.dumps(report,indent=2,allow_nan=False,default=str))
for key in ['comparison','official_gates','paired_parent_comparison','training_summary','window_metrics','saved_book_contribution_diagnostics']:
 print(key);print(pd.DataFrame(report[key]).to_string(index=False))


comparison
 cost_bps  sessions start_signal  last_exit  terminal_wealth  return_pct  cagr_pct  annualized_log_volatility_pct  max_drawdown_pct  qlib_max_drawdown_without_initial_mark_pct  worst_day_pct  cvar5_pct  worst_5d_pct   turnover  trade_days  sum_session_fee_rates  benchmark_return_pct  benchmark_cagr_pct  benchmark_max_drawdown_pct  benchmark_cvar5_pct  benchmark_worst_day_pct  benchmark_worst_5d_pct  log_excess_sum  relative_wealth  relative_return_pct  annualized_log_excess  annualized_relative_return_pct  relative_max_drawdown_pct  up_day_conditional_relative_return_pct  down_day_conditional_relative_return_pct  conditional_returns_are_noncontiguous_statistics                                                              risk_policy  annualization_sessions                             variant           segment
     10.0      1762   2016-01-04 2023-01-04         4.320293  332.029286 23.279439                      30.533249        -47.373291                                  -47

In [2]:
# Additional attribution of already-saved startup observations. No new policy.
with R.uri_context(reference['recorder_uri']):
 rec=R.get_recorder(experiment_name=plan['experiment'],recorder_id=reference['recorder_id'])
 folds=get('inputs/development_fold_inputs.pkl');folds.update(get('inputs/diagnostic_fold_inputs.pkl'))
 startup_training=[]
 for year in sorted(y for name,y in folds if name=='holdings_only'):
  audit=get(f'search_trials/holdings_only/{year}/ensemble/audit.pkl')
  dates=folds[('holdings_only',year)]['rows']
  for cost in (10,25):
   own=audit['training_paths'][cost];parent=audit['parent_training_paths'][cost]
   excess=np.log(own['net_equity'])-np.log(parent['net_equity'])
   different=np.flatnonzero(own['net_equity']!=parent['net_equity'])
   target=np.asarray(own['target_weights'])
   startup_training.append({'fit_year':year,'cost_bps':cost,'sessions':len(dates),
    'first_training_signal':str(dates.signal_date.iloc[0].date()),'first_training_trade':str(dates.trade_date.iloc[0].date()),
    'total_log_increment_vs_parent':float(excess.sum()),'first_observation_log_increment':float(excess[0]),
    'remaining_saved_observations_log_increment':float(excess[1:].sum()),
    'different_net_multiplier_rows':different.tolist(),'trade_rows':np.flatnonzero(own['turnover']>1e-12).tolist(),
    'non_full_SOXX_target_rows':np.flatnonzero(np.any(target!=[0.,1.,0.],axis=1)).tolist(),
    'scope':'Saved training book contribution versus its saved parent, no restarted path.'})
 startup_rows=[]
 for segment,variant in [('development','holdings_only'),('reused_diagnostic','holdings_only_annual_refit')]:
  part=paths[(paths.segment==segment)&(paths.variant==variant)&(paths.cost_bps==25)&(paths.seed=='ensemble')].sort_values('signal_date').reset_index(drop=True)
  benchmark=paths[(paths.segment==segment)&(paths.cost_bps==25)&(paths.seed=='benchmark')].sort_values('signal_date').reset_index(drop=True)
  for i in range(min(12,len(part))):
   row=part.iloc[i];base=benchmark.iloc[i]
   startup_rows.append({'segment':segment,'row':i,'signal_date':str(row.signal_date.date()),'trade_date':str(row.trade_date.date()),
    'soxl_weight':float(row.soxl_weight),'soxx_weight':float(row.soxx_weight),'cash_weight':float(row.cash_weight),
    'strategy_return_pct':100*float(row.strategy_return),'benchmark_return_pct':100*float(base.strategy_return),
    'log_excess':float(np.log1p(row.strategy_return)-np.log1p(base.strategy_return)),'fees':float(row.fees),
    'turnover':float(row.turnover),'exact_hold_support':bool(row.exact_hold_support),'exact_zero_order':bool(row.exact_zero_order)})
report['holding_only_saved_training_startup']=startup_training
report['holding_only_saved_market_startup']=startup_rows
Path('/tmp/phase9_economic_audit.json').write_text(json.dumps(report,indent=2,allow_nan=False,default=str))
print('SAVED HOLDINGS-ONLY TRAINING STARTUP')
for row in startup_training:
 if row['cost_bps']==25:
  print({k:v for k,v in row.items() if k not in ['non_full_SOXX_target_rows','scope']})
print(pd.DataFrame(startup_rows).to_string(index=False))


SAVED HOLDINGS-ONLY TRAINING STARTUP
{'fit_year': 2016, 'cost_bps': 25, 'sessions': 1256, 'first_training_signal': '2011-01-03', 'first_training_trade': '2011-01-04', 'total_log_increment_vs_parent': 0.0035823472415485447, 'first_observation_log_increment': 0.004675331310501038, 'remaining_saved_observations_log_increment': -0.001092984068952493, 'different_net_multiplier_rows': [0, 1, 2], 'trade_rows': [0, 1, 2]}
{'fit_year': 2017, 'cost_bps': 25, 'sessions': 1508, 'first_training_signal': '2011-01-03', 'first_training_trade': '2011-01-04', 'total_log_increment_vs_parent': 0.003622956719221383, 'first_observation_log_increment': 0.0047283017029119785, 'remaining_saved_observations_log_increment': -0.0011053449836905954, 'different_net_multiplier_rows': [0, 1, 2], 'trade_rows': [0, 1, 2]}
{'fit_year': 2018, 'cost_bps': 25, 'sessions': 1759, 'first_training_signal': '2011-01-03', 'first_training_trade': '2011-01-04', 'total_log_increment_vs_parent': 0.003246269773570765, 'first_observat

# 第九轮：搜索确实改善了训练目标，仍没有稳定市场超额

证据仅来自完整且FINISHED的市场Recorder **8696cd2152df4ffbbb5ac6692b139ef9**。父模型/搜索合成预检Recorder **a0cbd66785b44ccc8ab2080b36026195** 是单独的工程/优化证据；其F约0.650不是市场收益。本次说明只读取封存对象和已有账本作加法归因，没有新模型、新目标、额外策略回放或门槛修改。2023以后仍为已查看历史；没有新前向证据。

可执行只读复核：[phase9_economic_audit.ipynb](/tmp/phase9_economic_audit.ipynb)。完整指标、来源SHA与已有账本的起始贡献在`/tmp/phase9_economic_audit.json`。

## 原定门槛的结论

**三个组全部不合格。** 原排序选中`holdings_only`作诊断，并非通过资格。开发期是2016–2022信号年，共1,762个区间，首交易2016-01-05，末次2023-01-04开盘结算。

|组|相对SOXX终值10/25bp|正超额年份10/25bp|MaxDD10/25bp|CVaR5 25bp|交易天数 / 总换手25bp|
|---|---:|---:|---:|---:|---:|
|holdings_only|1.014755 / 1.014742|1/7、1/7|−47.373% / −47.373%|−4.416%|3 / 1.006倍|
|price_holdings|1.385531 / 1.036266|3/7、3/7|−49.033% / −49.432%|−5.771%|1455 / 194.575倍|
|price_vol_holdings|1.556598 / 1.114310|3/7、3/7|−64.262% / −64.570%|−6.605%|1541 / 223.765倍|

同窗SOXX25bp MaxDD−47.373%、CVaR5−4.418%。两个条件组在10/25bp都失败于MaxDD/CVaR、正年份数量、年度中位数、两种区块bootstrap下界；高成本下删掉任一年稳健性也失败。price_vol在10bp的删一年门通过，最小剩余净对数超额为0.06009，是可计量的正值；但25bp降为−0.23036。25bp两种区块bootstrap最差年化对数下界为price约−21.70%、price_vol约−29.64%，均非浮点量级，且不改变总体失败。原有门槛没有被本说明调整。

## 选中的持仓状态组：可计量的起始收益，不是持续信号

25bp累计净对数超额为 **0.0146348032556**，对应相对终值多1.47424%。这个量本身不属于浮点噪声；但它**全部集中在2016初始建仓**：

- 首日目标56.3973%SOXX、43.6027%现金，策略−1.9765%，基准−3.5008%；当日对数超额+0.0156723442。
- 次日升至99.7882%SOXX，当日对数超额−0.0010320819。
- 第三日补至100%SOXX，额外费用造成−0.0000054591；以后1,759天收益与SOXX逐日相同。
- 第一条观察的贡献大于整段收益；其余已保存观察合计−0.0010375410。这是同一账本的加法拆分，不是删除第一天后另造策略。

每个训练年度都从同一个2011-01-04交易日的现金状态开始。检查全部11个已保存集合训练账本，25bp相对父模型的正对数增量仅约0.00193–0.00387，均出现在开头2–10行，之后与父模型收益相同；七个开发训练折只有头2–3行不同。完整训练目标把一次性入场收益计为正增量，搜索因此能在所有折改善F，同时并未建立持续择时能力。这是直接的保存账本证据，不是对市场模型的重训推断。

年度与数值边界：25bp下2017–2022各年的对数超额**精确为0**；10bp下2017–2020约为−1e−14，其余0。原门只数到1个正年份，没有被这些微小残差虚增。删掉2016后，25bp余量精确0，10bp约−3.5e−14，所以删一年门合理失败。holdings_only最差bootstrap年化对数下界约 **−0.01484%**，也未通过；这不是一个显著正置信下界。三种子决策相同，不能读成三个独立市场样本。

2023以后，年更与冻结参数两条已保存路径的整段净收益相同，相对SOXX终值仅 **1.0008175**（25bp）。正增量仍来自2023-01-04入场第一天的少持SOXX；其余已保存观察合计对数超额−0.00114921。2024–2026年度超额为0。这进一步说明优势是现金起步时的暂态，不是多年度持续信号。

## 条件组：相对父模型有增量，但靠高幅度轮换承担更差尾部

同数据、同成本、同持仓会计的父模型对照已保存在正式Recorder中。25bp相对父模型终值增量：price **+2.489%**，price_vol **+7.887%**；10bp则为+26.443%/+38.102%。因此搜索不是简单退回父模型：全部63个开发种子都保留了搜索点，所有21个开发集合也选择搜索结果。问题转为费用后增益有限且跨年风险恶化，不是“优化器没有找到训练可行模型”。

price平均持仓为26.28%SOXL、0.49%SOXX、73.23%现金；price_vol为29.63%SOXL、0.066%SOXX、70.31%现金。持有大量平均现金没有保证危机日风险更小。25bp最差单日为−21.592%/−27.352%，父模型为−10.324%/−10.693%，SOXX为−9.325%；最差五日为−37.932%/−52.604%。

|组|25bp年化对数费用前贡献|费用项|净对数超额|相对父模型的总换手变化|
|---|---:|---:|---:|---:|
|price_holdings|+7.4330%|−6.9235%|+0.5095%|54.61→194.58倍|
|price_vol_holdings|+9.5157%|−7.9678%|+1.5480%|59.23→223.76倍|

这些是可相加的对数贡献，不是普通收益率加减。25bp费用消耗约93.1%/83.7%的费用前贡献。交易天数从父模型的1762天下降，但每次转仓规模更大，总换手分别约为父模型3.56/3.78倍；所以“少交易几天”没有实现更低交易成本。

两组25bp正年度都是2016、2018、2022。price逐年相对收益约+19.38%、−24.30%、+31.48%、−18.54%、−7.37%、−13.58%、+33.73%；price_vol约+40.30%、−20.44%、+34.78%、−6.26%、−26.33%、−9.64%、+18.70%。这些正负幅度很大，失败不是年度浮点噪声。price25bp整段净对数超额0.035624，其中第一条观察贡献0.035635，其余合计约−1.13e−5；尽管中途大起大落，净优势也对开始日期敏感。

beta/HAC不能挽救信息优势声明：price25bp实测beta约0.960，年化算术alpha点估计4.565%，95%区间[−12.883%,22.012%]；price_vol beta约1.131，alpha3.607%，区间[−18.571%,25.785%]。区间均跨零，且不包括全部模型选择不确定性。较高收益不能自动解释为可靠alpha；price的风险恶化也不能简单归因于平均beta升高。

## 精确HOLD与实际不下单应分开

三个**集合**开发路径的单独HOLD专家支持天数均为0。holdings_only的1759个零订单日来自已经持有纯SOXX；两个条件集合的零订单日分别307、188，这些零订单均不是单独HOLD专家触发；已处于现金或某资产顶点时，即使输出该资产也可不下单。

price的seed41独立路径确有14个HOLD日，其余条件种子为0；集合同时要求组件在共享持仓上形成一致HOLD，不能把单种子次数当成集合次数。price_vol有188个精确零订单日，但报告不交易日为221天，另33天是低于既有1e−12报告阈值的微小订单；不应混称为精确零订单。这里沿用原记录定义，没有另设交易带。

## 用户8月3日窗口

正式保存窗口按实际交易日起算：2026-08-03至09-25开盘，38个区间，首信号07-31。只允许开发阶段选出的holdings_only进入后续诊断，不能事后改选其他组。

|模式/起点|10bp策略 / SOXX收益|25bp策略 / SOXX收益|交易天数|
|---|---:|---:|---:|
|年更，持续持仓|15.843% / 15.843%|15.843% / 15.843%|0|
|冻结，持续持仓|15.843% / 15.843%|15.843% / 15.843%|0|
|年更，8/3现金起步|12.541% / 15.727%|12.373% / 15.554%|7|
|冻结，8/3现金起步|12.936% / 15.727%|12.769% / 15.554%|8|

持续持仓已经是SOXX，所以窗口内完全跟随基准，无额外超额。现金起步则落后约2.8–3.2个百分点，说明训练及2016入场时有利的过渡方式不能保证换一个起点仍有利。所有上述窗口MaxDD约−10.984%，与SOXX相同。该研究账本不是用户真实持仓；不能与截至09-30收盘的旧窗口混比。

## 本轮实际解决与未解决的事

解决：固定父模型编码器后，登记的33坐标COBYLA能够在合成和实际训练账本上找到原风险门可行的F改进；不依赖继承或回退冒充搜索成功。正式记录75个新搜索、109070次唯一双成本评估，另有63个开发缓存父模型与12个确实缺失外层父模型的旧协议重建（1440步旧Adam），这些计算预算不能混为同一种迭代。

未解决：稳定性、费用与样本外尾部。持仓状态组主要利用固定训练现金起点；条件组学到大幅SOXL/现金轮换，训练风险约束未稳定传递到下一年。更强的优化器和表达能力并未创造可靠的新市场信息。当前证据不能承诺稳定未来超额，不能作为生产切换或实盘收益证明。原门、排序及未晋升决定均保持不变。
